# Train INR Currency Counter on Kaggle (YOLOv8x)

Kaggle is a better fit than Colab for this because a long training run can survive a disconnect.

## Why Kaggle beats Colab here
- **Commit mode runs headless for up to 12h** with your browser closed — a dropped connection can't kill it (this is the fix for the Colab disconnect you hit).
- **30 GPU hours/week** on the free tier (T4 x2 or P100).
- The dataset is uploaded **once** as a Kaggle Dataset and reused across runs — no re-uploading each session.

## One-time setup
1. Locally, build the dataset: `python merge_datasets.py`, then zip the `data/merged` folder into **`merged.zip`**.
2. On Kaggle: left sidebar **Datasets → New Dataset →** upload `merged.zip` (Kaggle auto-extracts it). Name it e.g. `inr-merged`.
3. Open this notebook on Kaggle (**Create → Notebook → File → Import** this `.ipynb`).
4. Right sidebar: **Add Input →** your `inr-merged` dataset.
5. Right sidebar: **Session options → Accelerator → GPU T4 x2** (or P100), and **Internet: On** (needs phone verification once).

## To run without babysitting (the important part)
Click **Save Version → Save & Run All (Commit)**. Kaggle then runs every cell on its own servers for up to 12h and stores the output — you can close the tab entirely. When it finishes, grab `best.pt` from the notebook's **Output** tab.

## Step 1: Install / upgrade Ultralytics
Kaggle ships PyTorch already; we just need a current Ultralytics. (Requires Internet: On.)

In [ ]:
!pip install -q -U ultralytics

## Step 2: Verify the GPU
Confirm Kaggle attached a GPU. If this prints `False`, set Accelerator to GPU in the right sidebar and restart.

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA (GPU) available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  [{i}] {torch.cuda.get_device_name(i)}")

## Step 3: Locate the dataset and copy it to a writable folder
`/kaggle/input` is **read-only**, but YOLO needs to write `.cache` files next to the labels. So we auto-find `data.yaml` under your added dataset and copy the whole tree into `/kaggle/working/data` (writable).

In [ ]:
import os, glob, shutil

# Find data.yaml anywhere under the attached Kaggle datasets
candidates = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
if not candidates:
    raise FileNotFoundError(
        'No data.yaml found under /kaggle/input. Add your merged dataset via '
        '"Add Input" in the right sidebar (it should contain data.yaml + train/ + valid/).'
    )

src_yaml = candidates[0]
src_root = os.path.dirname(src_yaml)
print(f'Found dataset at: {src_root}')

dst_root = '/kaggle/working/data'
if os.path.exists(dst_root):
    shutil.rmtree(dst_root)
shutil.copytree(src_root, dst_root)
print(f'Copied dataset to writable path: {dst_root}')
print(os.listdir(dst_root))

## Step 4: Fix data.yaml paths for Kaggle
Your merged `data.yaml` hardcodes a Windows `path:`. Rewrite it to the Linux copy and make sure it points at `train/images` and `valid/images`.

In [ ]:
import yaml

yaml_path = os.path.join(dst_root, 'data.yaml')
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = dst_root
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg.pop('test', None)  # merge_datasets.py only produces train + valid

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print('Updated data.yaml:')
print(open(yaml_path).read())

# Sanity check that images actually exist
for split in ['train', 'valid']:
    n = len(glob.glob(os.path.join(dst_root, split, 'images', '*')))
    print(f'{split}: {n} images')

## Step 5: Train YOLOv8x
Ultralytics auto-downloads `yolov8x.pt` on first use (Internet: On). Outputs go to `/kaggle/working/models`.

**If you hit the 12h commit limit before it finishes**, re-run in commit mode and set `resume=True` below — it picks up from `last.pt` automatically.

**If you get a CUDA out-of-memory error**, lower `batch` (16 → 8 → 4).

In [ ]:
from ultralytics import YOLO

PROJECT = '/kaggle/working/models'
NAME = 'inr_counter_x'
RESUME = False  # set True to continue an interrupted run from last.pt

last_ckpt = os.path.join(PROJECT, NAME, 'weights', 'last.pt')
if RESUME and os.path.exists(last_ckpt):
    print(f'Resuming from {last_ckpt}')
    model = YOLO(last_ckpt)
    results = model.train(resume=True)
else:
    model = YOLO('yolov8x.pt')
    results = model.train(
        data=yaml_path,
        epochs=80,
        imgsz=640,
        batch=16,            # drop to 8 or 4 on CUDA OOM
        workers=4,
        project=PROJECT,
        name=NAME,
        device=0,            # use single GPU; set device=[0,1] to use T4 x2
        degrees=10.0,        # random rotation — notes are rarely perfectly flat
        shear=2.0,
        perspective=0.0005,  # slight perspective warp for camera angles
        mixup=0.15,          # helps with overlapping notes
        patience=20,         # early stop if no improvement
    )

print('Training complete.')

## Step 6: Surface the trained weights in the Output
Copy `best.pt` to the top of `/kaggle/working` so it's easy to find and download from the **Output** tab. After a committed run, download it and drop it into your local `models/best.pt` for `live_count.py`.

In [ ]:
best = os.path.join(PROJECT, NAME, 'weights', 'best.pt')
if os.path.exists(best):
    shutil.copy2(best, '/kaggle/working/best.pt')
    size_mb = os.path.getsize(best) / 1e6
    print(f'best.pt ({size_mb:.1f} MB) is ready at /kaggle/working/best.pt')
    print('Download it from the Output tab, then save locally as models/best.pt')
else:
    print(f'Could not find {best} — check the training logs above.')